# Four-hypothesis diagnostic sweep for the oscillatory non-potential game

[Open in Colab](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Ver3/notebooks/oscillatory_nonpotential_four_hypothesis_sweeps.ipynb)

This notebook diagnoses *why* DTB accuracy may deteriorate as the spatial frequency increases. It uses the reusable objects in `DTB_Ver3` and imports the tangent construction, truncated-SVD projection, and accumulated-state update directly from `DTB_Ver3.dtb`.

The game is

\[
b_\omega(x)=d(x)+q_\omega(x),\qquad
d(x)=\begin{pmatrix}-\kappa x_1\\-\kappa x_2\end{pmatrix},\qquad
q_\omega(x)=\begin{pmatrix}A\sin(\omega x_2)\\-A\sin(\omega x_1)\end{pmatrix},
\]

with \(\omega\in\{\pi,4\pi,8\pi,16\pi\}\). The four sections test:

1. insufficient tangent-space representation;
2. finite-sample under-resolution;
3. oscillatory information in weak singular directions;
4. amplification of small local errors by the game dynamics.

All functions defined below have mathematical docstrings stating the object they compute. The supplied problem does not prescribe \(A\) and \(\kappa\), so both remain controls and default to \(A=\kappa=1\).


In [ ]:
from pathlib import Path
import csv
import math
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate the package in a local/PACE checkout. Clone the branch only in Colab.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import OscillatoryNonpotentialGame, ResidualMMNN
from DTB_Ver3.dtb import (
    dtb_step,
    flat_parameters,
    project_velocity,
    subset_tangent_selection,
)
from DTB_Ver3.utils import resolve_device, resolve_dtype, to_numpy, warmup_cuda

package_dir = repo_root / 'DTB_Ver3'
output_dir = package_dir / 'results' / 'oscillatory_nonpotential_four_hypotheses'
output_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_dir)


## Global controls shared by all four hypotheses

The reference and training samples are independent. Float64 is intentional because Hypothesis 3 tests SVD thresholds down to `1e-10`. The MMNN is an identity-initialized residual map; its frozen random features define the tangent basis, while the trainable mixing coordinates define `theta`.


In [ ]:
SEED = 2026
KAPPA = 1.0
AMPLITUDE = 1.0
FREQUENCY_MULTIPLES = (1, 4, 8, 16)
OMEGAS = tuple(multiplier * np.pi for multiplier in FREQUENCY_MULTIPLES)

DEVICE_NAME = 'auto'
DTYPE_NAME = 'float64'
DEVICE = resolve_device(DEVICE_NAME)
DTYPE = resolve_dtype(DTYPE_NAME)
warmup_cuda(DEVICE, DTYPE)

MMNN_WIDTH = 12
MMNN_RANK = 12
MMNN_DEPTH = 3
ACTIVATION = 'tanh'
JACOBIAN_CHUNK = 512

# Hypothesis 1: independent validation set and independent feature-basis resets.
N_REPRESENTATION_REFERENCE = 5000
RESET_MODEL_SEEDS = (SEED + 101, SEED + 102, SEED + 103)
REPRESENTATION_SVD_RTOL = 1e-10

# Hypothesis 2: nested training samples, repeated over independent seeds.
N_SAMPLING_REFERENCE = 20000
SAMPLE_SIZES = (500, 1000, 2000, 5000, 10000, 20000)
SAMPLING_SEEDS = (SEED + 201, SEED + 202, SEED + 203)
SAMPLING_SVD_RTOL = 1e-8

# Hypothesis 3: the requested relative SVD cutoff sweep.
SVD_TOLERANCES = (1e-2, 1e-4, 1e-6, 1e-8, 1e-10)

# Hypothesis 4: matched DTB, Euler, and refined RK4 trajectories.
N_DYNAMIC = 500
DYNAMIC_FINAL_TIME = 0.2
BASE_STEP_SIZE = 0.01
STEP_SIZES = tuple(BASE_STEP_SIZE / divisor for divisor in (1, 2, 4, 8))
RK4_REFERENCE_STEP = 0.00025
DYNAMIC_SVD_RTOL = 1e-8

print({
    'device': str(DEVICE),
    'dtype': str(DTYPE),
    'omega_over_pi': FREQUENCY_MULTIPLES,
    'sample_sizes': SAMPLE_SIZES,
    'svd_tolerances': SVD_TOLERANCES,
    'step_sizes': STEP_SIZES,
})


## Mathematical helper functions

These helpers organize experiments only. The neural parameter Jacobian, least-squares projection, and DTB update themselves remain the package implementations imported from `DTB_Ver3.dtb`.


In [ ]:
def build_identity_mmnn(seed):
    r"""Construct the residual solution map

    .. math:: T_{\theta_0}(z)=z+M_{\theta_0}(z)=z.

    The output mixing layer is zeroed, while the MMNN feature matrices are
    reproducibly drawn from ``seed``. Changing the seed therefore resets the
    neural tangent feature basis without changing the represented identity map.
    """
    torch.manual_seed(int(seed))
    return ResidualMMNN(
        2,
        width=MMNN_WIDTH,
        rank=MMNN_RANK,
        depth=MMNN_DEPTH,
        activation=ACTIVATION,
        dtype=DTYPE,
        zero_init_output=True,
    ).to(DEVICE)


def draw_uniform(count, seed):
    r"""Draw the Monte Carlo quadrature cloud

    .. math:: z_i\stackrel{\mathrm{iid}}{\sim}U([-1,1]^2),\quad i=1,\ldots,N.

    Sampling occurs on CPU with a private generator and the complete tensor is
    then transferred to the selected device, preserving reproducibility.
    """
    generator = torch.Generator().manual_seed(int(seed))
    points = 2.0 * torch.rand((int(count), 2), generator=generator, dtype=DTYPE) - 1.0
    return points.to(DEVICE)


def full_tangent_matrix(model, points):
    r"""Compute the sampled neural parameter Jacobian

    .. math:: \mathcal J(\theta,Z)=N^{-1/2}
       [J_\theta(z_1)^\top,\ldots,J_\theta(z_N)^\top]^\top.

    ``subset_tangent_selection`` supplies the unnormalized stacked matrix from
    ``DTB_Ver3.dtb``; this helper returns that matrix together with the flat
    parameter vector, structure, and all coordinate indices.
    """
    theta, structure = flat_parameters(model)
    selected = torch.arange(theta.numel(), device=theta.device)
    _, matrix = subset_tangent_selection(
        theta,
        selected,
        points,
        model,
        structure,
        chunk_size=JACOBIAN_CHUNK,
    )
    return theta, structure, selected, matrix.detach()


def active_tangent_columns(matrix, selected, relative_tolerance=1e-13):
    r"""Remove structurally zero columns from ``J``.

    Mathematically this preserves ``range(J)`` while replacing the coordinate
    representation by the nonzero column set

    .. math:: S=\{j:\|J_{:,j}\|_2>\varepsilon\max_\ell\|J_{:,\ell}\|_2\}.

    Zero columns occur at identity initialization because earlier MMNN layers
    are screened by the zero output layer; discarding them avoids meaningless
    zero singular values without changing the tangent space.
    """
    norms = torch.linalg.vector_norm(matrix, dim=0)
    threshold = relative_tolerance * norms.max()
    keep = norms > threshold
    if not keep.any():
        raise FloatingPointError('the tangent matrix has no active columns')
    return matrix[:, keep], selected[keep]


def relative_l2(approximation, target):
    r"""Compute ``||approximation-target||_2 / ||target||_2``.

    This is the common dimensionless error used for representation, sampling,
    and truncated-SVD diagnostics.
    """
    denominator = torch.linalg.vector_norm(target).clamp_min(torch.finfo(target.dtype).tiny)
    return float((torch.linalg.vector_norm(approximation - target) / denominator).item())


def projection_metrics(tangent_matrix, target_velocity, rtol):
    r"""Compute the truncated-SVD projection of ``q`` onto ``range(J)``.

    The package solves

    .. math:: \alpha^*=\arg\min_\alpha\|J\alpha-q\|_2^2

    and this helper reports ``E_repr``, ``R_osc=1-E_repr^2``, and
    ``||alpha*||_2`` without reimplementing the DTB least-squares solver.
    """
    projection = project_velocity(tangent_matrix, target_velocity, rtol=rtol)
    error = float(projection.relative_residual.item())
    return {
        'projection': projection,
        'representation_error': error,
        'captured_energy': 1.0 - error**2,
        'alpha_norm': float(torch.linalg.vector_norm(projection.alpha).item()),
    }


def rk4_flow(game, state, time_value, interval, maximum_step):
    r"""Approximate the exact interval flow ``Phi_h(state)`` by refined RK4.

    Each substep applies

    .. math:: x^+=x+\delta(k_1+2k_2+2k_3+k_4)/6

    to ``dx/dt=b_omega(x,t)``. The number of equal substeps is chosen so that
    ``delta <= maximum_step`` and the interval endpoint is hit exactly.
    """
    substeps = max(1, int(math.ceil(float(interval) / float(maximum_step) - 1e-12)))
    delta = float(interval) / substeps
    value = state.detach().clone()
    for substep in range(substeps):
        time = float(time_value) + substep * delta
        k1 = game.velocity(value, time)
        k2 = game.velocity(value + 0.5 * delta * k1, time + 0.5 * delta)
        k3 = game.velocity(value + 0.5 * delta * k2, time + 0.5 * delta)
        k4 = game.velocity(value + delta * k3, time + delta)
        value = value + (delta / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
    return value.detach()


def paired_rms(first, second):
    r"""Compute ``sqrt(mean_i ||first_i-second_i||_2^2)``."""
    return float((first - second).square().sum(dim=1).mean().sqrt().item())


def run_dynamic_diagnostic(game, model, theta_initial, structure, selected, initial, step_size):
    r"""Compute all Hypothesis-4 trajectories and local diagnostics.

    DTB uses ``X[k+1]=X[k]+h J[k] alpha[k]`` from ``dtb_step``. In parallel,
    Euler uses ``X_E[k+1]=X_E[k]+h b(X_E[k])`` and RK4 approximates the exact
    reference. At each step this returns ``E_proj``, the one-step error
    ``||X_DTB[k+1]-Phi_h(X_DTB[k])||_RMS``, DTB/RK4 and Euler/RK4 trajectory
    errors, and the logarithmic growth rate
    ``mu=lambda_max((Db+Db.T)/2)``.
    """
    steps = int(round(DYNAMIC_FINAL_TIME / step_size))
    if not np.isclose(steps * step_size, DYNAMIC_FINAL_TIME, atol=1e-12, rtol=0.0):
        raise ValueError('DYNAMIC_FINAL_TIME must be an integer multiple of every step size')

    theta = theta_initial.detach().clone()
    dtb_state = initial.detach().clone()
    euler_state = initial.detach().clone()
    reference_state = initial.detach().clone()
    fixed_labels = initial.detach().clone()

    times = [0.0]
    dtb_trajectory_error = [0.0]
    euler_trajectory_error = [0.0]
    projection_error = []
    relative_projection_error = []
    one_step_error = []
    mean_growth = []
    max_growth = []
    positive_growth_fraction = []

    for step in range(steps):
        time_value = step * step_size
        old_dtb = dtb_state
        target = game.velocity(old_dtb, time_value)
        theta, dtb_state, projection = dtb_step(
            theta,
            selected,
            old_dtb,
            target,
            model,
            structure,
            step_size=step_size,
            chunk_size=JACOBIAN_CHUNK,
            svd_rtol=DYNAMIC_SVD_RTOL,
            tangent_inputs=fixed_labels,
        )

        # Accurate one-step flow from the current DTB state isolates local error.
        local_reference = rk4_flow(
            game, old_dtb, time_value, step_size, RK4_REFERENCE_STEP
        )
        one_step_error.append(paired_rms(dtb_state, local_reference))

        # Matched global references start from the same initial particle labels.
        euler_state = (
            euler_state + step_size * game.velocity(euler_state, time_value)
        ).detach()
        reference_state = rk4_flow(
            game, reference_state, time_value, step_size, RK4_REFERENCE_STEP
        )

        projection_error.append(float(projection.rms_residual.item()))
        relative_projection_error.append(float(projection.relative_residual.item()))
        dtb_trajectory_error.append(paired_rms(dtb_state, reference_state))
        euler_trajectory_error.append(paired_rms(euler_state, reference_state))

        growth = game.symmetric_growth_rate(old_dtb)
        mean_growth.append(float(growth.mean().item()))
        max_growth.append(float(growth.max().item()))
        positive_growth_fraction.append(float((growth > 0).to(DTYPE).mean().item()))
        times.append((step + 1) * step_size)

    return {
        'times': np.asarray(times),
        'projection_times': np.asarray(times[:-1]),
        'projection_error': np.asarray(projection_error),
        'relative_projection_error': np.asarray(relative_projection_error),
        'one_step_error': np.asarray(one_step_error),
        'dtb_trajectory_error': np.asarray(dtb_trajectory_error),
        'euler_trajectory_error': np.asarray(euler_trajectory_error),
        'mean_growth': np.asarray(mean_growth),
        'max_growth': np.asarray(max_growth),
        'positive_growth_fraction': np.asarray(positive_growth_fraction),
        'dtb_final': to_numpy(dtb_state),
        'euler_final': to_numpy(euler_state),
        'rk4_final': to_numpy(reference_state),
    }


def save_table(path, columns, rows):
    r"""Serialize a finite-dimensional diagnostic table ``{row_j}`` to CSV."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(columns)
        writer.writerows(rows)
    return target


# Hypothesis 1 — Insufficient tangent-space representation

We evaluate only the oscillatory target \(q_\omega\) on a large independent validation cloud. For a fixed MMNN feature basis,

\[
E_{\mathrm{repr}}(\omega)=
\frac{\|\mathcal J_{\mathrm{ref}}\alpha_{\mathrm{ref}}-q_{\omega,\mathrm{ref}}\|_2}
{\|q_{\omega,\mathrm{ref}}\|_2},\qquad
R_{\mathrm{osc}}=1-E_{\mathrm{repr}}^2.
\]

The comparison uses one frozen basis, three independently reset feature bases of the same dimension, and an enriched space formed by concatenating their active tangent directions. The concatenated space is a diagnostic direct-sum upper bound; it is not substituted for the single-network tangent in the DTB dynamics. All frequencies use exactly the same validation samples and bases. A large error even for the enriched space supports a genuine representation limitation; a substantial reduction after reset or enrichment supports a basis-resolution limitation.


In [ ]:
representation_points = draw_uniform(N_REPRESENTATION_REFERENCE, SEED + 10)

# Build one frozen identity map and several reset identity maps. Every Jacobian
# is computed by DTB_Ver3.dtb.subset_tangent_selection.
print('H1: building frozen validation tangent', flush=True)
base_model = build_identity_mmnn(SEED + 100)
base_theta, base_structure, base_all_indices, base_full_matrix = full_tangent_matrix(
    base_model, representation_points
)
base_matrix, base_active_indices = active_tangent_columns(
    base_full_matrix, base_all_indices
)

reset_matrices = []
for reset_seed in RESET_MODEL_SEEDS:
    print(f'H1: building reset validation tangent, seed={reset_seed}', flush=True)
    reset_model = build_identity_mmnn(reset_seed)
    _, _, reset_indices, reset_full = full_tangent_matrix(reset_model, representation_points)
    reset_active, _ = active_tangent_columns(reset_full, reset_indices)
    reset_matrices.append(reset_active)

# Concatenation represents the enriched union of independently reset bases.
enriched_matrix = torch.cat([base_matrix, *reset_matrices], dim=1)
print({
    'full_trainable_coordinates': int(base_theta.numel()),
    'frozen_active_dimension': int(base_matrix.shape[1]),
    'reset_active_dimensions': [int(matrix.shape[1]) for matrix in reset_matrices],
    'enriched_dimension': int(enriched_matrix.shape[1]),
})

h1_rows = []
for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
    target = game.oscillatory_velocity(representation_points)

    frozen = projection_metrics(base_matrix, target, REPRESENTATION_SVD_RTOL)
    h1_rows.append([
        multiplier, 'frozen', 0, base_matrix.shape[1],
        frozen['representation_error'], frozen['captured_energy'], frozen['alpha_norm'],
    ])

    for reset_index, matrix in enumerate(reset_matrices, start=1):
        reset = projection_metrics(matrix, target, REPRESENTATION_SVD_RTOL)
        h1_rows.append([
            multiplier, 'reset', reset_index, matrix.shape[1],
            reset['representation_error'], reset['captured_energy'], reset['alpha_norm'],
        ])

    enriched = projection_metrics(enriched_matrix, target, REPRESENTATION_SVD_RTOL)
    h1_rows.append([
        multiplier, 'enriched', 0, enriched_matrix.shape[1],
        enriched['representation_error'], enriched['captured_energy'], enriched['alpha_norm'],
    ])

h1_columns = (
    'omega_over_pi', 'basis_configuration', 'replicate', 'active_dimension',
    'E_repr', 'R_osc', 'alpha_norm',
)
h1_path = save_table(output_dir / 'hypothesis_1_representation.csv', h1_columns, h1_rows)
print('saved:', h1_path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
for configuration, color, marker in (
    ('frozen', 'tab:blue', 'o'),
    ('reset', 'tab:orange', 's'),
    ('enriched', 'tab:green', '^'),
):
    means, stds, captured = [], [], []
    for multiplier in FREQUENCY_MULTIPLES:
        rows = [row for row in h1_rows if row[0] == multiplier and row[1] == configuration]
        means.append(np.mean([row[4] for row in rows]))
        stds.append(np.std([row[4] for row in rows]))
        captured.append(np.mean([row[5] for row in rows]))
    axes[0].errorbar(
        FREQUENCY_MULTIPLES, means, yerr=stds, marker=marker,
        linewidth=1.6, capsize=3, label=configuration, color=color,
    )
    axes[1].plot(FREQUENCY_MULTIPLES, captured, marker=marker,
                 linewidth=1.6, label=configuration, color=color)

axes[0].set(xlabel=r'$\omega/\pi$', ylabel=r'$E_{\mathrm{repr}}$',
            title='Oscillatory representation error')
axes[1].set(xlabel=r'$\omega/\pi$', ylabel=r'$R_{\mathrm{osc}}$',
            title='Captured oscillatory energy')
for axis in axes:
    axis.set_xscale('log', base=2)
    axis.grid(True, alpha=0.3)
    axis.legend()
h1_figure = output_dir / 'hypothesis_1_representation.png'
fig.savefig(h1_figure, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', h1_figure)


# Hypothesis 2 — Finite samples under-resolve the oscillatory field

The frozen active basis from Hypothesis 1 is held fixed. For each training size and seed, `project_velocity` computes \(\widehat\alpha_N\) on the training cloud. An independent reference cloud then measures

\[
E_N(\omega)=\frac{\|\mathcal J_{\mathrm{ref}}\widehat\alpha_N-q_{\omega,\mathrm{ref}}\|_2}{\|q_{\omega,\mathrm{ref}}\|_2},\qquad
E_{\mathrm{sample}}=
\frac{\|\mathcal J_{\mathrm{ref}}(\widehat\alpha_N-\alpha_{\mathrm{ref}})\|_2}{\|q_{\omega,\mathrm{ref}}\|_2}.
\]

We also compare the empirical normal-equation objects

\[
G_N=N^{-1}J_N^\top J_N,\qquad c_N=N^{-1}J_N^\top q_N
\]

with their large-sample counterparts. A flat validation error as \(N\) grows indicates a representation floor; a decreasing excess error indicates sampling under-resolution.


In [ ]:
sampling_reference = draw_uniform(N_SAMPLING_REFERENCE, SEED + 20)
_, sampling_reference_matrix = subset_tangent_selection(
    base_theta,
    base_active_indices,
    sampling_reference,
    base_model,
    base_structure,
    chunk_size=JACOBIAN_CHUNK,
)
G_reference = (
    sampling_reference_matrix.T @ sampling_reference_matrix / N_SAMPLING_REFERENCE
)

# Reference coefficients and c_ref depend on omega; G_ref does not.
h2_reference = {}
for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
    target_reference = game.oscillatory_velocity(sampling_reference)
    projection_reference = project_velocity(
        sampling_reference_matrix,
        target_reference,
        rtol=SAMPLING_SVD_RTOL,
    )
    c_reference = (
        sampling_reference_matrix.T @ target_reference.reshape(-1)
        / N_SAMPLING_REFERENCE
    )
    h2_reference[multiplier] = (
        target_reference,
        projection_reference.alpha,
        c_reference,
    )

h2_rows = []
maximum_sample_size = max(SAMPLE_SIZES)
for sample_seed in SAMPLING_SEEDS:
    # Nested prefixes reduce Monte Carlo noise when comparing adjacent N values.
    print(f'H2: building nested training tangent, seed={sample_seed}', flush=True)
    training_points = draw_uniform(maximum_sample_size, sample_seed)
    _, training_matrix_all = subset_tangent_selection(
        base_theta,
        base_active_indices,
        training_points,
        base_model,
        base_structure,
        chunk_size=JACOBIAN_CHUNK,
    )
    for sample_size in SAMPLE_SIZES:
        points = training_points[:sample_size]
        matrix = training_matrix_all[: 2 * sample_size]
        G_sample = matrix.T @ matrix / sample_size
        E_G = relative_l2(G_sample, G_reference)

        for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
            game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
            target = game.oscillatory_velocity(points)
            fitted = project_velocity(matrix, target, rtol=SAMPLING_SVD_RTOL)
            target_reference, alpha_reference, c_reference = h2_reference[multiplier]
            validation_prediction = (
                sampling_reference_matrix @ fitted.alpha
            ).reshape_as(target_reference)
            E_N = relative_l2(validation_prediction, target_reference)
            sampling_difference = (
                sampling_reference_matrix @ (fitted.alpha - alpha_reference)
            ).reshape_as(target_reference)
            E_sample = float(
                torch.linalg.vector_norm(sampling_difference).div(
                    torch.linalg.vector_norm(target_reference)
                ).item()
            )
            c_sample = matrix.T @ target.reshape(-1) / sample_size
            E_c = relative_l2(c_sample, c_reference)
            h2_rows.append([
                multiplier, sample_size, sample_seed,
                E_N, E_sample, E_G, E_c,
            ])

h2_columns = ('omega_over_pi', 'sample_size', 'seed', 'E_N', 'E_sample', 'E_G', 'E_c')
h2_path = save_table(output_dir / 'hypothesis_2_sample_size.csv', h2_columns, h2_rows)
print('saved:', h2_path)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for multiplier in FREQUENCY_MULTIPLES:
    summaries = []
    for sample_size in SAMPLE_SIZES:
        rows = [row for row in h2_rows if row[0] == multiplier and row[1] == sample_size]
        summaries.append([
            sample_size,
            *[np.mean([row[column] for row in rows]) for column in range(3, 7)],
            *[np.std([row[column] for row in rows]) for column in range(3, 7)],
        ])
    summaries = np.asarray(summaries)
    for plot_index, (axis, label) in enumerate(zip(
        axes.flat,
        (r'$E_N$', r'$E_{\mathrm{sample}}$', r'$E_G$', r'$E_c$'),
    )):
        axis.errorbar(
            summaries[:, 0], summaries[:, 1 + plot_index],
            yerr=summaries[:, 5 + plot_index], marker='o', capsize=2,
            label=fr'$\omega={multiplier}\pi$',
        )
        axis.set(xlabel='training sample size N', ylabel=label)
        axis.set_xscale('log')
        axis.set_yscale('log')
        axis.grid(True, alpha=0.3, which='both')
axes[0, 0].set_title('Validation projection error')
axes[0, 1].set_title('Excess sampling error')
axes[1, 0].set_title('Gram-matrix quadrature error')
axes[1, 1].set_title('Tangent/forcing correlation error')
axes[0, 0].legend()
h2_figure = output_dir / 'hypothesis_2_sample_size.png'
fig.savefig(h2_figure, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', h2_figure)


# Hypothesis 3 — Oscillatory information lies in weak singular directions

For the normalized frozen tangent matrix \(\mathcal J=U\Sigma V^\top\), define \(\beta_i=u_i^\top q_\omega\). At relative cutoff \(\tau\), the notebook records

\[
E_{\mathrm{TSVD}}=\frac{\|q_\omega-P_{r_\tau}q_\omega\|_2}{\|q_\omega\|_2},\quad
\kappa_{\mathrm{eff}}=\frac{\sigma_1}{\sigma_{r_\tau}},\quad
W_\tau=\frac{\sum_{\sigma_i/\sigma_1<\tau}|\beta_i|^2}{\|q_\omega\|_2^2}.
\]

The Picard plot shows \(\sigma_i\), \(|\beta_i|\), and \(|\beta_i|/\sigma_i\). Strong error reduction accompanied by coefficient-norm growth as \(\tau\) decreases supports the weak-direction hypothesis.


In [ ]:
# Reuse the independent representation cloud and its frozen active tangent matrix.
normalized_matrix = base_matrix / math.sqrt(N_REPRESENTATION_REFERENCE)
left, singular_values, right_h = torch.linalg.svd(normalized_matrix, full_matrices=False)
sigma_ratio = singular_values / singular_values[0]

h3_rows = []
h3_picard = {}
for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
    target = game.oscillatory_velocity(representation_points).reshape(-1)
    normalized_target = target / math.sqrt(N_REPRESENTATION_REFERENCE)
    target_norm = torch.linalg.vector_norm(normalized_target)
    beta = left.T @ normalized_target
    h3_picard[multiplier] = {
        'sigma_ratio': to_numpy(sigma_ratio),
        'beta_ratio': to_numpy(beta.abs() / target_norm),
        'coefficient_ratio': to_numpy(beta.abs() / singular_values),
    }

    for tolerance in SVD_TOLERANCES:
        retained = sigma_ratio > tolerance
        if not retained.any():
            raise FloatingPointError(f'no singular modes retained at tau={tolerance:g}')
        alpha = right_h[retained].T @ (beta[retained] / singular_values[retained])
        projected = left[:, retained] @ beta[retained]
        E_tsvd = float(
            (torch.linalg.vector_norm(normalized_target - projected) / target_norm).item()
        )
        weak = sigma_ratio < tolerance
        W_tau = float(beta[weak].square().sum().div(target_norm.square()).item())
        kappa_effective = float((singular_values[0] / singular_values[retained][-1]).item())
        h3_rows.append([
            multiplier, tolerance, int(retained.sum().item()), E_tsvd,
            float(torch.linalg.vector_norm(alpha).item()), W_tau, kappa_effective,
        ])

h3_columns = (
    'omega_over_pi', 'svd_rtol', 'retained_rank', 'E_TSVD',
    'alpha_norm', 'W_tau', 'kappa_effective',
)
h3_path = save_table(output_dir / 'hypothesis_3_svd_tolerance.csv', h3_columns, h3_rows)
print('saved:', h3_path)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for multiplier in FREQUENCY_MULTIPLES:
    rows = np.asarray([row[1:] for row in h3_rows if row[0] == multiplier], dtype=float)
    tolerance = rows[:, 0]
    axes[0, 0].loglog(tolerance, rows[:, 2], 'o-', label=fr'$\omega={multiplier}\pi$')
    axes[0, 1].loglog(tolerance, rows[:, 3], 'o-')
    axes[1, 0].loglog(tolerance, np.maximum(rows[:, 4], 1e-18), 'o-')
    axes[1, 1].loglog(tolerance, rows[:, 5], 'o-')
axes[0, 0].set(xlabel=r'$\tau$', ylabel=r'$E_{\mathrm{TSVD}}$', title='TSVD projection error')
axes[0, 1].set(xlabel=r'$\tau$', ylabel=r'$\|\alpha^*\|_2$', title='Coefficient norm')
axes[1, 0].set(xlabel=r'$\tau$', ylabel=r'$W_\tau$', title='Target energy in discarded weak modes')
axes[1, 1].set(xlabel=r'$\tau$', ylabel=r'$\kappa_{\mathrm{eff}}$', title='Effective condition number')
for axis in axes.flat:
    axis.invert_xaxis()
    axis.grid(True, alpha=0.3, which='both')
axes[0, 0].legend()
h3_figure = output_dir / 'hypothesis_3_svd_tolerance.png'
fig.savefig(h3_figure, dpi=180, bbox_inches='tight')
plt.show()

# Discrete Picard plot for the hardest frequency.
picard = h3_picard[max(FREQUENCY_MULTIPLES)]
mode = np.arange(1, len(picard['sigma_ratio']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axes[0].semilogy(mode, picard['sigma_ratio'], '.-')
axes[1].semilogy(mode, np.maximum(picard['beta_ratio'], 1e-20), '.-')
axes[2].semilogy(mode, np.maximum(picard['coefficient_ratio'], 1e-20), '.-')
axes[0].set(xlabel='singular-mode index i', ylabel=r'$\sigma_i/\sigma_1$', title='Singular spectrum')
axes[1].set(xlabel='singular-mode index i', ylabel=r'$|\beta_i|/\|q\|$', title='Target coefficient')
axes[2].set(xlabel='singular-mode index i', ylabel=r'$|\beta_i|/\sigma_i$', title='Solution coefficient')
for axis in axes:
    axis.grid(True, alpha=0.3)
picard_path = output_dir / 'hypothesis_3_picard_16pi.png'
fig.savefig(picard_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', h3_figure)
print('saved:', picard_path)


# Hypothesis 4 — Small velocity errors are amplified by the game dynamics

The package DTB update is compared with explicit Euler and refined RK4 from identical initial particles. For each \((\omega,h)\), the notebook records the instantaneous projection residual, global trajectory errors, and the one-step error

\[
E_{\mathrm{step}}(t_k)=
\left[\frac1N\sum_i\|X^{i}_{\mathrm{DTB}}(t_k+h)-
\Phi_h(X^{i}_{\mathrm{DTB}}(t_k))\|_2^2\right]^{1/2}.
\]

It also evaluates the largest eigenvalue of the symmetric vector-field Jacobian,

\[
\mu_\omega(x)=\lambda_{\max}\!\left(\frac{Db_\omega+Db_\omega^\top}{2}\right)
=-\kappa+\frac{A\omega}{2}|\cos(\omega x_2)-\cos(\omega x_1)|.
\]

Small projection error with large long-time error correlated with positive \(\mu_\omega\) supports dynamical amplification. Strong improvement of both Euler and DTB as \(h\) decreases indicates temporal-resolution error.


In [ ]:
dynamic_initial = draw_uniform(N_DYNAMIC, SEED + 30)
dynamic_selected = torch.arange(base_theta.numel(), device=DEVICE)
dynamic_runs = {}
h4_rows = []

for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
    for step_size in STEP_SIZES:
        print(f'H4: omega={multiplier}pi, h={step_size:g}', flush=True)
        diagnostic = run_dynamic_diagnostic(
            game,
            base_model,
            base_theta,
            base_structure,
            dynamic_selected,
            dynamic_initial,
            step_size,
        )
        dynamic_runs[(multiplier, step_size)] = diagnostic
        h4_rows.append([
            multiplier,
            step_size,
            diagnostic['dtb_trajectory_error'][-1],
            diagnostic['euler_trajectory_error'][-1],
            diagnostic['projection_error'].mean(),
            diagnostic['relative_projection_error'].mean(),
            diagnostic['one_step_error'].mean(),
            diagnostic['mean_growth'].mean(),
            diagnostic['max_growth'].max(),
            diagnostic['positive_growth_fraction'].mean(),
        ])
        np.savez_compressed(
            output_dir / f'h4_omega_{multiplier}pi_h_{step_size:g}.npz',
            **diagnostic,
        )

h4_columns = (
    'omega_over_pi', 'step_size', 'final_DTB_RK4_RMS', 'final_Euler_RK4_RMS',
    'mean_projection_RMS', 'mean_relative_projection_error',
    'mean_one_step_RMS', 'mean_mu', 'max_mu', 'positive_mu_fraction',
)
h4_path = save_table(output_dir / 'hypothesis_4_dynamics.csv', h4_columns, h4_rows)
print('saved:', h4_path)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for multiplier in FREQUENCY_MULTIPLES:
    rows = np.asarray([row[1:] for row in h4_rows if row[0] == multiplier], dtype=float)
    step = rows[:, 0]
    order = np.argsort(step)
    label = fr'$\omega={multiplier}\pi$'
    axes[0, 0].loglog(step[order], rows[order, 1], 'o-', label=label)
    axes[0, 1].loglog(step[order], rows[order, 2], 'o-', label=label)
    axes[1, 0].loglog(step[order], rows[order, 4], 'o-', label=label)
    axes[1, 1].loglog(step[order], rows[order, 5], 'o-', label=label)
axes[0, 0].set(xlabel='step size h', ylabel='final paired RMS', title='DTB versus RK4')
axes[0, 1].set(xlabel='step size h', ylabel='final paired RMS', title='Euler versus RK4')
axes[1, 0].set(xlabel='step size h', ylabel='relative residual', title='Mean tangent projection error')
axes[1, 1].set(xlabel='step size h', ylabel='one-step RMS', title='Mean DTB one-step error')
for axis in axes.flat:
    axis.grid(True, alpha=0.3, which='both')
axes[0, 0].legend()
h4_figure = output_dir / 'hypothesis_4_step_size.png'
fig.savefig(h4_figure, dpi=180, bbox_inches='tight')
plt.show()

# Time-resolved view at the highest frequency.
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for step_size in STEP_SIZES:
    run = dynamic_runs[(max(FREQUENCY_MULTIPLES), step_size)]
    label = f'h={step_size:g}'
    axes[0, 0].semilogy(run['times'], np.maximum(run['dtb_trajectory_error'], 1e-18), label=label)
    axes[0, 1].semilogy(run['times'], np.maximum(run['euler_trajectory_error'], 1e-18), label=label)
    axes[1, 0].semilogy(run['projection_times'], np.maximum(run['one_step_error'], 1e-18), label=label)
    axes[1, 1].plot(run['projection_times'], run['mean_growth'], label=label)
axes[0, 0].set(xlabel='time', ylabel='paired RMS', title=r'DTB trajectory error, $\omega=16\pi$')
axes[0, 1].set(xlabel='time', ylabel='paired RMS', title=r'Euler trajectory error, $\omega=16\pi$')
axes[1, 0].set(xlabel='time', ylabel='one-step RMS', title='Local DTB flow error')
axes[1, 1].set(xlabel='time', ylabel=r'mean $\mu_\omega$', title='Mean symmetric growth rate')
for axis in axes.flat:
    axis.grid(True, alpha=0.3)
axes[0, 0].legend()
h4_time_path = output_dir / 'hypothesis_4_16pi_trajectories.png'
fig.savefig(h4_time_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', h4_figure)
print('saved:', h4_time_path)


# Four-hypothesis evidence summary

The following ratios are compact evidence checks, not automatic proofs. Read them together with the full curves:

- Hypothesis 1: whether enrichment lowers the high-frequency representation error.
- Hypothesis 2: whether increasing `N` lowers excess sampling error.
- Hypothesis 3: whether reducing `tau` lowers TSVD error while increasing coefficient norm.
- Hypothesis 4: whether time-step refinement lowers trajectory error and whether trajectory error co-varies with the symmetric growth rate.


In [ ]:
highest = max(FREQUENCY_MULTIPLES)

frozen_high = np.mean([row[4] for row in h1_rows if row[0] == highest and row[1] == 'frozen'])
enriched_high = np.mean([row[4] for row in h1_rows if row[0] == highest and row[1] == 'enriched'])

h2_small = np.mean([
    row[4] for row in h2_rows if row[0] == highest and row[1] == min(SAMPLE_SIZES)
])
h2_large = np.mean([
    row[4] for row in h2_rows if row[0] == highest and row[1] == max(SAMPLE_SIZES)
])

h3_loose = next(row for row in h3_rows if row[0] == highest and row[1] == max(SVD_TOLERANCES))
h3_tight = next(row for row in h3_rows if row[0] == highest and row[1] == min(SVD_TOLERANCES))

h4_coarse = next(row for row in h4_rows if row[0] == highest and row[1] == max(STEP_SIZES))
h4_fine = next(row for row in h4_rows if row[0] == highest and row[1] == min(STEP_SIZES))
growth_values = np.asarray([row[7] for row in h4_rows])
trajectory_values = np.asarray([row[2] for row in h4_rows])
growth_error_correlation = float(np.corrcoef(growth_values, trajectory_values)[0, 1])

checks = [
    ['H1', 'enriched/frozen E_repr at 16pi', enriched_high / frozen_high],
    ['H2', 'large-N/small-N E_sample at 16pi', h2_large / h2_small],
    ['H3', 'tight/loose E_TSVD at 16pi', h3_tight[3] / h3_loose[3]],
    ['H3', 'tight/loose alpha norm at 16pi', h3_tight[4] / h3_loose[4]],
    ['H4', 'fine/coarse DTB final RMS at 16pi', h4_fine[2] / h4_coarse[2]],
    ['H4', 'correlation(mean mu, final DTB RMS)', growth_error_correlation],
]
checks_path = save_table(
    output_dir / 'four_hypothesis_evidence_checks.csv',
    ('hypothesis', 'diagnostic_ratio_or_correlation', 'value'),
    checks,
)
for hypothesis, diagnostic, value in checks:
    print(f'{hypothesis}: {diagnostic} = {value:.4e}')
print('saved:', checks_path)


# Export all tables, arrays, and figures

The final archive is written beside the result directory. On PACE, download it through JupyterLab's file browser after the last cell finishes.


In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_dir),
    'zip',
    root_dir=output_dir,
))
print('result archive:', archive_path)
